<a href="https://colab.research.google.com/github/raj-Tk/AI-Infrastructure-Index/blob/main/indxx_assignment_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Infrastructure Enablers Index

This notebook implements the Indxx campus hiring assignment using a tighter AI infrastructure theme.

## Objective
Build a systematic, rules-based thematic index and backtest it against both a thematic benchmark and the overall market.

## Data Source and Assumptions
- Prices and volumes come from `yfinance`.
- Adjusted close is used for return calculation.
- Historical free-float market cap is approximated with `shares_outstanding * price`.
- When historical shares are unavailable, the latest available share count is used as a fallback.
- Quarterly rebalancing is applied on the first trading day of each calendar quarter.
- Between rebalances, weights drift naturally with price movement.
- The revenue exposure test is implemented through a curated universe of companies whose businesses are materially tied to AI compute, cloud capacity, chips, networking, or data center infrastructure.

## Rulebook

**Theme & Rationale**

The AI Infrastructure Enablers Index is designed to capture companies whose revenue is meaningfully driven by the buildout of AI compute and cloud capacity. The theme is investable because AI adoption requires spending across chips, servers, networking, cloud platforms, and physical data center infrastructure. This creates a cleaner and more defensible ETF narrative than a broader “cloud + semis” basket.

**Eligibility Criteria**
- U.S.-listed companies or liquid U.S.-traded ADRs.
- Company must belong to at least one of the following buckets:
  GPU and AI chip designers, semiconductor manufacturing and equipment, hyperscale cloud platforms, networking vendors, server and power infrastructure vendors, or data center REITs.
- Company should have meaningful business exposure to AI infrastructure buildout. For practical implementation with free data, this is approximated through a curated universe intended to represent names with at least roughly 30% strategic or revenue exposure to AI infrastructure.
- Minimum market capitalization: USD 10 billion.

**Liquidity Filter**
- Minimum 3-month average daily traded value: USD 20 million.

**Number of Constituents**
- Hold up to 30 eligible stocks ranked by market cap proxy.

**Weighting Methodology**
- Equal weight at each quarterly rebalance to reduce concentration risk in mega-cap names.

**Rebalancing & Reconstitution**
- Quarterly reconstitution and quarterly rebalancing.
- Constituents and target weights are recomputed on each rebalance date.

In [ ]:
# Uncomment if you need to install dependencies in a fresh environment.
# %pip install yfinance pandas numpy matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# Parameters
START_DATE = "2021-01-01"
END_DATE = None  # set a date string like "2026-04-15" to freeze the backtest
LOOKBACK_DAYS = 63
MIN_MARKET_CAP = 10_000_000_000
MIN_ADTV = 20_000_000
MAX_CONSTITUENTS = 30
BASE_VALUE = 1000.0

THEMATIC_BENCHMARK = "SOXX"
MARKET_BENCHMARK = "SPY"

# Refined universe focused on AI infrastructure enablers.
UNIVERSE = [
    "NVDA", "AMD", "AVGO", "MRVL", "MU", "ARM", "MPWR", "ON", "GFS",
    "TSM", "ASML", "AMAT", "LRCX", "KLAC",
    "ANET", "CSCO",
    "SMCI", "DELL", "HPE", "VRT",
    "EQIX", "DLR",
    "MSFT", "AMZN", "GOOGL", "ORCL"
]

# Optional metadata for explaining the theme in the walkthrough.
THEME_BUCKETS = {
    "NVDA": "AI chips",
    "AMD": "AI chips",
    "AVGO": "AI chips / networking",
    "MRVL": "AI chips / networking",
    "MU": "memory",
    "ARM": "chip IP",
    "MPWR": "power management",
    "ON": "power / semis",
    "GFS": "foundry",
    "TSM": "foundry",
    "ASML": "semicap equipment",
    "AMAT": "semicap equipment",
    "LRCX": "semicap equipment",
    "KLAC": "semicap equipment",
    "ANET": "networking",
    "CSCO": "networking",
    "SMCI": "AI servers",
    "DELL": "servers",
    "HPE": "servers",
    "VRT": "power and cooling",
    "EQIX": "data center REIT",
    "DLR": "data center REIT",
    "MSFT": "hyperscaler",
    "AMZN": "hyperscaler",
    "GOOGL": "hyperscaler",
    "ORCL": "cloud infrastructure",
}

In [ ]:
def download_market_data(tickers, start_date, end_date=None):
    raw = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        available = sorted(set(raw.columns.get_level_values(0)))
        adj_close = pd.concat(
            {ticker: raw[ticker]["Adj Close"] for ticker in available if ("Adj Close" in raw[ticker].columns)},
            axis=1,
        )
        volume = pd.concat(
            {ticker: raw[ticker]["Volume"] for ticker in available if ("Volume" in raw[ticker].columns)},
            axis=1,
        )
    else:
        available = [tickers[0]]
        adj_close = raw[["Adj Close"]].rename(columns={"Adj Close": tickers[0]})
        volume = raw[["Volume"]].rename(columns={"Volume": tickers[0]})

    adj_close = adj_close.sort_index().dropna(how="all")
    volume = volume.reindex(adj_close.index).sort_index()
    missing = [ticker for ticker in tickers if ticker not in adj_close.columns]
    return adj_close, volume, available, missing


all_tickers = UNIVERSE + [THEMATIC_BENCHMARK, MARKET_BENCHMARK]
prices, volumes, available_tickers, missing_tickers = download_market_data(all_tickers, START_DATE, END_DATE)

active_universe = [ticker for ticker in UNIVERSE if ticker in prices.columns]
removed_universe = [ticker for ticker in UNIVERSE if ticker not in active_universe]

universe_prices = prices[active_universe].copy()
universe_volumes = volumes[active_universe].copy()
benchmark_prices = prices[[THEMATIC_BENCHMARK, MARKET_BENCHMARK]].copy()

print(f"Active universe size: {len(active_universe)}")
print(f"Removed tickers with no data: {removed_universe}")
print(f"Price history shape: {universe_prices.shape}")
print(f"Benchmark history shape: {benchmark_prices.shape}")
universe_prices.tail()

In [ ]:
def build_share_history(tickers, start_date, end_date=None):
    share_histories = {}
    static_shares = {}

    for ticker in tickers:
        tk = yf.Ticker(ticker)

        shares_series = None
        try:
            shares_series = tk.get_shares_full(start=start_date, end=end_date)
        except Exception:
            shares_series = None

        if shares_series is not None and not shares_series.empty:
            shares_series = shares_series.sort_index()
            shares_series.index = pd.to_datetime(shares_series.index).tz_localize(None)
            share_histories[ticker] = shares_series.astype(float)

        fallback = None
        for key in ("sharesOutstanding", "impliedSharesOutstanding"):
            try:
                info = tk.fast_info if hasattr(tk, "fast_info") else {}
                fallback = info.get(key)
            except Exception:
                fallback = None
            if fallback:
                break

        if fallback is None:
            try:
                info = tk.info
                fallback = info.get("sharesOutstanding") or info.get("impliedSharesOutstanding")
            except Exception:
                fallback = None

        static_shares[ticker] = float(fallback) if fallback else np.nan

    return share_histories, static_shares


share_histories, static_shares = build_share_history(active_universe, START_DATE, END_DATE)
pd.Series(static_shares).dropna().sort_values(ascending=False).head()

In [ ]:
def get_shares_on_date(ticker, date, share_histories, static_shares):
    series = share_histories.get(ticker)
    if series is not None and not series.empty:
        valid = series.loc[series.index <= date]
        if not valid.empty:
            return float(valid.iloc[-1])
        return float(series.iloc[0])
    return float(static_shares.get(ticker, np.nan))


def get_rebalance_dates(trading_index, lookback_days):
    calendar = pd.date_range(trading_index.min(), trading_index.max(), freq="QS")
    rebalance_dates = []
    for dt in calendar:
        eligible = trading_index[trading_index >= dt]
        if len(eligible) == 0:
            continue
        candidate = eligible[0]
        if trading_index.get_loc(candidate) >= lookback_days:
            rebalance_dates.append(candidate)
    return pd.DatetimeIndex(sorted(set(rebalance_dates)))


def select_constituents_on_date(date, prices, volumes, share_histories, static_shares, weighting="equal"):
    loc = prices.index.get_loc(date)
    if loc < LOOKBACK_DAYS:
        return pd.Series(dtype=float), pd.DataFrame()

    price_window = prices.iloc[loc - LOOKBACK_DAYS:loc]
    volume_window = volumes.iloc[loc - LOOKBACK_DAYS:loc]
    adtv = (price_window * volume_window).mean()
    current_prices = prices.iloc[loc]

    rows = []
    for ticker in prices.columns:
        px = current_prices.get(ticker, np.nan)
        adv = adtv.get(ticker, np.nan)
        shares = get_shares_on_date(ticker, date, share_histories, static_shares)
        market_cap = px * shares if pd.notna(px) and pd.notna(shares) else np.nan

        rows.append(
            {
                "ticker": ticker,
                "theme_bucket": THEME_BUCKETS.get(ticker, "other"),
                "price": px,
                "adtv": adv,
                "shares_outstanding": shares,
                "market_cap_proxy": market_cap,
            }
        )

    eligibility = pd.DataFrame(rows).set_index("ticker")
    eligibility = eligibility[
        eligibility["price"].notna()
        & (eligibility["adtv"] >= MIN_ADTV)
        & (eligibility["market_cap_proxy"] >= MIN_MARKET_CAP)
    ].sort_values("market_cap_proxy", ascending=False)

    selected = eligibility.head(MAX_CONSTITUENTS).copy()
    if selected.empty:
        return pd.Series(dtype=float), eligibility

    if weighting == "equal":
        weights = pd.Series(1 / len(selected), index=selected.index)
    elif weighting == "market_cap":
        weights = selected["market_cap_proxy"] / selected["market_cap_proxy"].sum()
    else:
        raise ValueError("weighting must be 'equal' or 'market_cap'")

    return weights.sort_index(), eligibility


rebalance_dates = get_rebalance_dates(universe_prices.index, LOOKBACK_DAYS)
rebalance_dates[:5], rebalance_dates[-5:]

In [ ]:
def run_backtest(prices, volumes, benchmark_prices, weighting="equal"):
    index_levels = pd.Series(index=prices.index, dtype=float)
    benchmark_levels = pd.DataFrame(index=prices.index, columns=benchmark_prices.columns, dtype=float)
    weights_by_date = {}
    selections = {}
    turnover_rows = []

    returns = prices.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    benchmark_returns = benchmark_prices.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

    current_weights = None

    for i, rebalance_date in enumerate(rebalance_dates):
        target_weights, eligible_table = select_constituents_on_date(
            rebalance_date, prices, volumes, share_histories, static_shares, weighting=weighting
        )
        if target_weights.empty:
            continue

        next_rebalance = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        segment_dates = prices.index[(prices.index >= rebalance_date) & (prices.index <= next_rebalance)]
        if len(segment_dates) < 2:
            continue

        selections[rebalance_date] = eligible_table.loc[target_weights.index]
        weights_by_date[rebalance_date] = target_weights

        if index_levels.dropna().empty:
            index_levels.loc[rebalance_date] = BASE_VALUE
            for bench in benchmark_levels.columns:
                benchmark_levels.loc[rebalance_date, bench] = BASE_VALUE
            pre_rebalance_weights = pd.Series(0.0, index=target_weights.index)
        else:
            pre_rebalance_weights = current_weights.reindex(target_weights.index).fillna(0.0)

        turnover = 0.5 * (target_weights - pre_rebalance_weights).abs().sum()
        turnover_rows.append({"rebalance_date": rebalance_date, "turnover": turnover})

        current_weights = target_weights.copy()

        for prev_date, curr_date in zip(segment_dates[:-1], segment_dates[1:]):
            asset_returns = returns.loc[curr_date, current_weights.index].fillna(0.0)
            portfolio_return = float((current_weights * asset_returns).sum())

            prev_level = index_levels.loc[prev_date]
            if pd.isna(prev_level):
                prev_level = index_levels.ffill().loc[prev_date]
            index_levels.loc[curr_date] = prev_level * (1 + portfolio_return)

            for bench in benchmark_levels.columns:
                bench_prev = benchmark_levels.loc[prev_date, bench]
                if pd.isna(bench_prev):
                    bench_prev = benchmark_levels[bench].ffill().loc[prev_date]
                benchmark_levels.loc[curr_date, bench] = bench_prev * (1 + benchmark_returns.loc[curr_date, bench])

            gross = current_weights * (1 + asset_returns)
            gross_sum = gross.sum()
            current_weights = gross / gross_sum if gross_sum > 0 else current_weights

    result = pd.concat(
        [
            index_levels.ffill().rename("index_level"),
            benchmark_levels.ffill(),
        ],
        axis=1,
    ).dropna()

    weights_table = pd.DataFrame(weights_by_date).T.fillna(0.0).sort_index()
    turnover_table = pd.DataFrame(turnover_rows)
    return result, weights_table, turnover_table, selections


result_equal, weights_equal, turnover_equal, selections_equal = run_backtest(
    universe_prices, universe_volumes, benchmark_prices, weighting="equal"
)

result_equal.head()

In [ ]:
def annualized_return(series):
    years = (series.index[-1] - series.index[0]).days / 365.25
    total_return = series.iloc[-1] / series.iloc[0] - 1
    return (1 + total_return) ** (1 / years) - 1


def max_drawdown(series):
    return (series / series.cummax() - 1).min()


def compute_metrics(levels, turnover_table):
    metric_rows = [
        "Total Return (%)",
        "Annualized Return (%)",
        "Annualized Volatility (%)",
        "Sharpe Ratio",
        "Maximum Drawdown (%)",
        "Avg. Turnover per Rebalance (%)",
    ]
    metrics = pd.DataFrame(index=metric_rows)

    for column in levels.columns:
        daily_returns = levels[column].pct_change().dropna()
        total_return = levels[column].iloc[-1] / levels[column].iloc[0] - 1
        ann_return = annualized_return(levels[column])
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
        mdd = max_drawdown(levels[column])

        metrics[column] = [
            total_return * 100,
            ann_return * 100,
            ann_vol * 100,
            sharpe,
            mdd * 100,
            np.nan,
        ]

    metrics.loc["Avg. Turnover per Rebalance (%)", "index_level"] = turnover_table["turnover"].mean() * 100
    return metrics.rename(
        columns={
            "index_level": "AI Infra Index",
            THEMATIC_BENCHMARK: "SOXX",
            MARKET_BENCHMARK: "S&P 500 (SPY)",
        }
    )


metrics_equal = compute_metrics(result_equal, turnover_equal)
metrics_equal

In [ ]:
latest_selection = next(reversed(selections_equal))
latest_constituents = selections_equal[latest_selection].copy()
latest_constituents["theme_bucket"] = latest_constituents.index.map(THEME_BUCKETS)

display(latest_constituents[["theme_bucket", "price", "adtv", "market_cap_proxy"]].head(15))

print(f"Latest rebalance date: {latest_selection.date()}")
print(f"Constituent count: {len(latest_constituents)}")
print(f"Average turnover per rebalance: {turnover_equal['turnover'].mean() * 100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
result_equal.rename(
    columns={
        "index_level": "AI Infra Index",
        THEMATIC_BENCHMARK: "SOXX",
        MARKET_BENCHMARK: "S&P 500 (SPY)",
    }
).plot(ax=ax, linewidth=2)
ax.set_title("AI Infrastructure Index vs Benchmarks (Rebased to 1000)")
ax.set_xlabel("")
ax.set_ylabel("Index Level")
plt.show()

metrics_equal

In [ ]:
# Bonus sensitivity analysis: compare equal weight to market-cap weight
result_mcap, weights_mcap, turnover_mcap, selections_mcap = run_backtest(
    universe_prices, universe_volumes, benchmark_prices, weighting="market_cap"
)
metrics_mcap = compute_metrics(result_mcap, turnover_mcap)

sensitivity = pd.concat(
    [
        metrics_equal[["AI Infra Index"]].rename(columns={"AI Infra Index": "Equal Weight"}),
        metrics_mcap[["AI Infra Index"]].rename(columns={"AI Infra Index": "Market Cap Weight"}),
    ],
    axis=1,
)
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
comparison = pd.concat(
    [
        result_equal["index_level"].rename("Equal Weight"),
        result_mcap["index_level"].rename("Market Cap Weight"),
        result_equal[MARKET_BENCHMARK].rename("S&P 500 (SPY)"),
    ],
    axis=1,
).dropna()
comparison.plot(ax=ax, linewidth=2)
ax.set_title("Bonus Sensitivity Analysis")
ax.set_xlabel("")
ax.set_ylabel("Index Level")
plt.show()

## Decision Log

1. Narrowed the theme from broad “semiconductors and data centers” to “AI Infrastructure Enablers” to create a sharper and more defensible index narrative.
2. Used a curated universe instead of a broad sector screen because free public data does not provide a robust thematic revenue classification feed.
3. Removed weaker or less essential names from the original basket and kept a tighter set of companies directly tied to chips, cloud, networking, servers, power, and data centers.
4. Dropped `JNPR` after data quality issues in `yfinance`, which would weaken notebook reliability during evaluation.
5. Kept a USD 10 billion market cap floor to maintain investability and reduce noise from small-cap hardware names.
6. Kept a USD 20 million ADTV floor to support ETF-style implementation.
7. Used equal weight as the main methodology because market-cap weighting would make the index overly dominated by the largest hyperscalers and chip leaders.
8. Added both `SOXX` and `SPY` as benchmarks so performance can be framed relative to a relevant sector benchmark and the overall market.
9. Kept the bonus analysis focused on weighting methodology because it is easy to explain live and highlights the concentration trade-off clearly.
10. Documented the free-float market cap proxy explicitly because it is a reasonable but imperfect approximation when only free public data is allowed.

## Walkthrough Notes

- The main pitch is that AI adoption drives infrastructure spending across chips, networking, cloud, and data centers, so the index captures the enablers rather than end-user software beneficiaries.
- The cleanest rule to explain is the theme definition plus investability filters, followed by equal weighting to control concentration.
- If asked to modify the methodology live, switch `weighting="equal"` to `weighting="market_cap"` or adjust `MAX_CONSTITUENTS`, `MIN_ADTV`, or the universe list.